<a href="https://colab.research.google.com/github/kdeshmukh31-ux/RAG_Project_Kshitij/blob/main/RAG_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
!pip install langchain faiss-cpu sentence-transformers transformers pypdf

In [14]:
from google.colab import files
uploaded = files.upload()


Saving Employee_Handbook.pdf to Employee_Handbook.pdf
Saving IT_Security_Guidelines.pdf to IT_Security_Guidelines.pdf
Saving Company_PTO_Policy.pdf to Company_PTO_Policy.pdf


In [15]:
from pypdf import PdfReader

def load_pdf(path):
    text = ""
    reader = PdfReader(path)
    for page in reader.pages:
        text += page.extract_text() + "\n"
    return text

documents = []
for filename in uploaded.keys():
    text = load_pdf(filename)
    documents.append({"text": text, "source": filename})

len(documents)


3

In [16]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
)

chunks = []
for doc in documents:
    for chunk in splitter.split_text(doc["text"]):
        chunks.append({"text": chunk, "source": doc["source"]})

len(chunks)


3

In [17]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

texts = [c["text"] for c in chunks]
sources = [c["source"] for c in chunks]

embeddings = model.encode(texts)
len(embeddings)


3

In [18]:
import faiss
import numpy as np

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))


In [19]:
def retrieve(query, k=5, threshold=0.40):
    q_emb = model.encode([query])
    scores, ids = index.search(np.array(q_emb), k)

    results = []
    for score, idx in zip(scores[0], ids[0]):
        sim = 1 / (1 + score)
        if sim < threshold:
            continue
        results.append({
            "text": chunks[idx]["text"],
            "source": chunks[idx]["source"],
            "similarity": float(sim)
        })
    return results



In [20]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
llm_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")


In [21]:
def answer_question(query):
    results = retrieve(query)

    if len(results) == 0:
        return "I cannot answer this question based on the provided documents.", []

    context = "\n\n".join([r["text"] for r in results])

    prompt = f"""
You are a helpful assistant.
Use ONLY the context below to answer.
If the answer is not present, say "I cannot answer based on the documents."

Context:
{context}

Question: {query}
Answer:
"""

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)
    outputs = llm_model.generate(**inputs, max_new_tokens=150)
    ans = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return ans, results


In [22]:
query = "How many vacation days do employees get?"
answer, refs = answer_question(query)

print("Answer:\n", answer)
print("\nSources:")
for r in refs:
    print("-", r["source"])


Answer:
 18

Sources:
- Company_PTO_Policy.pdf
- Employee_Handbook.pdf


In [23]:
query = "Whom do we have to contact for applying PTO?"
answer, refs = answer_question(query)

print("Answer:\n", answer)
print("\nSources:")
for r in refs:
    print("-", r["source"], )



Answer:
 the reporting manager

Sources:
- Company_PTO_Policy.pdf


In [24]:
query = "can I share confidential company data externally"
answer, refs = answer_question(query)

print("Answer:\n", answer)
print("\nSources:")
for r in refs:
    print("-", r["source"])


Answer:
 No

Sources:
- IT_Security_Guidelines.pdf


In [25]:
query = "What is the weather tomorrow?"
answer, refs = answer_question(query)

print("Answer:\n", answer)
print("\nSources:")
for r in refs:
    print("-", r["source"])


Answer:
 I cannot answer this question based on the provided documents.

Sources:


In [1]:
import gradio as gr


def chat_fn(query):
    answer, retrieved = answer_question(query)

    # Build clean sources list
    source_list = ""
    for r in retrieved:
        source_list += f"- {r['source']}\n"

    # Build retrieved chunk preview
    retrieved_text = ""
    for r in retrieved:
        retrieved_text += f"\n\n---\n📄 **Source: {r['source']}**\n{r['text'][:500]}..."

    return answer, source_list, retrieved_text


with gr.Blocks() as demo:
    gr.Markdown("# 📘 RAG Q&A FOR COMPANY INTERNAL DOCUMENTS")

    question = gr.Textbox(label="Ask a Question", placeholder="Ask anything from your uploaded documents...")

    answer_box = gr.Textbox(label="Answer")
    sources_box = gr.Textbox(label="Sources Used (File Names)")

    ask_btn = gr.Button("Get Answer")

    ask_btn.click(
        fn=chat_fn,
        inputs=question,
        outputs=[answer_box, sources_box]
    )

demo.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ac8091bbaf4030e44d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
